In [16]:
%run Geom_Prolongation.ipynb
%run Particular_Distributions.ipynb
%run CartanGeometry.ipynb

In [17]:
g=Symp_symb(7)
D,D_JS,temp=Standard_Prenorm_distr(g,6,constant_coeff=true)
K=IndexedBase('K')

Jacobi id of weight 1 computed in time 1.0 sec
Jacobi id of weight 2 computed in time 1.0 sec
Jacobi id of weight 3 computed in time 1.0 sec
Jacobi id of weight 4 computed in time 1.0 sec
Jacobi id of weight 5 computed in time 1.0 sec
Jacobi id of weight 6 computed in time 1.0 sec


In [18]:
plus_val=6

In [19]:
def wght_converter(i,j,plus_val):
    """returns the first index k of D.basis such that D.basis[k] has weight
    wght(D.basis[i])+wght(D.basis[j])+plus_val"""
    w=g.basis[i+3].wght+g.basis[j+3].wght+plus_val
    for k in list(range(3,len(g.basis))).__reversed__():
        if g.basis[k].wght>=w: return k-3
    

In [20]:
def compute_ad_mat(u,D):
    if type(u)==Vector_Field:
        v=copy.deepcopy(u) 
    else: v=Vector_Field(u,D)
    r=[]
    for i in range(len(D.basis)):
        r.append(D.bracket(v,D.basis[i]).vec[3:len(D.basis[0].vec)])
    return Matrix(r).transpose()

In [21]:
ad_mats=[]
for i in range(len(D.basis)):
    ad_mats.append(compute_ad_mat(D.basis[i],D))

In [22]:
def ad_mat(u,D):
    if type(u)==Vector_Field:
        v=u.vec
    else: v=u
    r=zeros(len(D.basis))
    for i in range(len(D.basis)):
        try: r+=v[i+3]*ad_mats[i]
        except:
            M=zeros(*shape(ad_mats[i]))
            for j in range(shape(ad_mats[i])[0]): M[j,j]=v[i+3]
            r+=M*ad_mats[i]
    return r

In [23]:
for i in range(len(D.basis)):
    for j in range(i+1,len(D.basis)):
        ai,aj=[D.basis[i],D.basis[j]]
        Ai,Aj=[ad_mats[i],ad_mats[j]]
        M=(ad_mat(D.bracket(ai,aj),D)-(Ai*Aj-Aj*Ai)).xreplace(sol_dict)
        for a in range(len(D.basis)):
            for b in range(len(D.basis)):
                if g.basis[a+3].wght+g.basis[i+3].wght+g.basis[j+3].wght+plus_val<g.basis[b+3].wght:
                    M[b,a]=0
        M=Matrix(expand(M))
        if M!=zeros(*shape(M)): print((i,j))

NameError: name 'sol_dict' is not defined

In [ ]:
i,j=(0,4)
ai,aj=[D.basis[i],D.basis[j]]
Ai,Aj=[ad_mats[i],ad_mats[j]]
M=(ad_mat(D.bracket(ai,aj),D)-(Ai*Aj-Aj*Ai)).xreplace(sol_dict)
M=Matrix(expand(M))


In [ ]:
for a in range(len(D.basis)):
    for b in range(len(D.basis)):
        if g.basis[a+3].wght+g.basis[i+3].wght+g.basis[j+3].wght+plus_val<g.basis[b+3].wght:
            M[b,a]=0
if M!=zeros(*shape(M)): print((i,j))

In [ ]:
def wght_of_entry(a,b):
    g.basis[a+3].wght+g.basis[b+3.wght]

In [ ]:
expand(M)

In [ ]:
D_bookmark=copy.deepcopy(D)
# D=copy.deepcopy(D_bookmark)

In [ ]:
sol_dict={}
skipped_eqs=set()
for i in range(len(D.basis)):
    for j in range(i+1,len(D.basis)):
        print((i,j))
        time0=time.time()
        ai,aj=[D.basis[i],D.basis[j]]
        Ai,Aj=[ad_mats[i],ad_mats[j]]
        M=(ad_mat(D.bracket(ai,aj),D)-(Ai*Aj-Aj*Ai)).xreplace(sol_dict)
        time1=time.time()
        print('    Mats computed in',hrs_min_sec(time1-time0))
        for a in M:
            a=expand(a)
            if a!=0:
                time2=time.time()
                t=find_a_linear_term(a)
                time3=time.time()
                if t==None: 
                    print('        No linear term found in',hrs_min_sec(time3-time2))
                    skipped_eqs.add(a)
                else: 
                    print('        Linear term found in',hrs_min_sec(time3-time2))
                    sol_dict[t]=solve(a,t)[0]
                    little_dict={t:sol_dict[t]}
                    time4=time.time()
                    print('        Solving complete in',hrs_min_sec(time4-time3))
                    D.curv=D.curv.xreplace(little_dict)
                    print('        Substitution complete in',hrs_min_sec(time.time()-time4))
        print((i,j),'complete in',hrs_min_sec(time.time()-time0))


In [ ]:
skipped_eqs

In [ ]:
[0][10]

In [10]:
skipped_eqs=list(skipped_eqs)

del_inds=[]
for i in range(len(skipped_eqs)):
    if expand(skipped_eqs[i].subs(sol_dict))==0: del_inds.append(i)

del_inds.reverse()
for i in del_inds: 
    del skipped_eqs[i]

In [ ]:
skipped_eqs

In [ ]:
for i in range(len(D.basis)):
    for j in range(len(D.basis)):
        ai,aj=[D.basis[i],D.basis[j]]
        Ai,Aj=[ad_mats[i],ad_mats[j]]
        M=(compute_ad_mat(D.bracket(ai,aj),D)-(Ai*Aj-Ai*Aj)).subs(sol_dict)
        if M!=zeros(*shape(M)):
            display((i,j))
            display(M)

In [ ]:
ell=[0,0,0]
for i in range(len(D.basis)):
    exec('x{j}=symbols("x{j}")'.format(j=i))
    exec('ell[{j}+3]=x{j}'.format(j=i))
u=Vector_Field(ell)

In [ ]:
exp_u=exp(compute_ad_mat(u))

In [ ]:
M=(exp_u-Id(*shape(exp_u)[0])).inv()

In [ ]:
v=Vector_Field(([0,0,0,1,0,0,0,0,0,0,0]),D)
r=[]
for i in range(len(D.basis)):
    r.append(D.bracket(v,D.basis[i]).vec[3:len(g.basis)])
display(Matrix(r).transpose())

## Realizing Brandon's First Symmetry Algebra

In [25]:
a0,a1,a2,b=symbols('a0,a1,a2,b')
e,e1,e2,e3,e4,e5,e6,n=symbols('e,e1,e2,e3,e4,e5,e6,n')
basis=[e,e1,e2,e3,e4,e5,e6,n]
mult_table=Matrix([[0,-Rational(1,2)*e1,-Rational(1,2)*e2,-Rational(1,2)*e3,-Rational(1,2)*e4,
                    -Rational(1,2)*e5,-Rational(1,2)*e6,-n],
                    [Rational(1,2)*e1,0,0,0,0,0,n,0],
                    [Rational(1,2)*e2,0,0,0,0,-n,0,0],
                    [Rational(1,2)*e3,0,0,0,n,0,a0*n,0],
                    [Rational(1,2)*e4,0,0,-n,0,-a0*n,-2*a1*n,0],
                    [Rational(1,2)*e5,0,n,0,a0*n,0,(a0**2-a2-b)*n,0],
                    [Rational(1,2)*e6,-n,0,-a0*n,2*a1*n,-(a0**2-a2-b)*n,0,0],
                    [n,0,0,0,0,0,0,0]])

In [26]:
def quick_coordinatize(expr,basis):
    r=[]
    for i in range(len(basis)):
        s={basis[j]:0 for j in range(len(basis)) if j!=i}
        s[basis[i]]=1
        r.append(expr.xreplace(s))
    return r

ad_mats=[]
for i in range(len(basis)):
    r=[]
    for j in range(len(basis)):
        r.append(quick_coordinatize(mult_table[i,j],basis))
    ad_mats.append(Matrix(r).transpose())

def ad_mat(v):
    r=zeros(len(v))
    for i in range(len(v)):
        try: r+=v[i]*ad_mats[i]
        except:
            M=zeros(*shape(ad_mats[i]))
            for j in range(shape(ad_mats[i])[0]): M[j,j]=v[i]
            r+=M*ad_mats[i]
    return r

def Jacobi_check(mult_table,basis):
    for i in range(len(basis)):
        for j in range(len(basis)):
            for k in range(len(basis)):
                L=[i,j,k,i,j,k]
                r=0
                for ell in range(3):
                    i0,i1,i2=L[ell:ell+3]
                    v=quick_coordinatize(mult_table[i0,i1],basis)
                    for a in range(len(v)):
                        r+=v[a]*mult_table[a,i2]
                if simplify(r)!=0: print('Failure at',(i,j,k))

In [ ]:
# Constructing u and x
for i in range(len(basis)): 
    exec('u_{j}=symbols("u_{j}")'.format(j=i))
    exec('x_{j}=symbols("x_{j}")'.format(j=i))
u=0
for i in range(1,len(basis)): exec('u+=u_{j}*basis[{j}]'.format(j=i))
x=[]
for i in range(len(basis)): exec('x.append(x_{j})'.format(j=i))
x=Matrix(x)

s={}
for i in range(len(basis)): exec('s[x_{j}]=0'.format(j=i))

In [ ]:
ad_u=ad_mat(quick_coordinatize(u,basis))
Ad_u=simplify(exp(ad_u))
Ad_inv_u=Matrix(exp(-ad_u))

# Tunnel formula
# Check the convention for the bernoulli numbers...
p_u=(eye(len(basis))+Rational(1,2)*ad_u)
# for i in range(1,2):
#     p_u+=(-1)**(i+1)*bernoulli(i)/factorial(2*i)*ad_u**(2*i)
p_u=simplify(Matrix(p_u[1:len(basis),1:len(basis)])*Matrix(Ad_inv_u[1:len(basis),0:len(basis)]))

p_x_u=simplify(p_u*x)

for i in range(shape(p_u)[1]):
    exec('X_{j}=p_u.col({j})'.format(j=i))

In [ ]:
def temp_ad(X,Y):
    r=zeros(*shape(X))
    for i in range(len(X)):
        y=[u_1,u_2,u_3,u_4,u_5,u_6,u_7][i]
        for j in range(len(X)):
            r[j]+=X[i]*diff(Y[j],y)-Y[i]*diff(X[j],y)
    return r

In [ ]:
for i in range(len(basis)):
    for j in range(len(basis)):
        r1=p_u*Matrix(quick_coordinatize(mult_table[i,j],basis))
        exec('r2=simplify(temp_ad(X_{k1},X_{k2}))'.format(k1=i,k2=j))
        if simplify(r2+r1)!=zeros(*shape(r1)):print(i,j)

In [ ]:
symb_reps=[]
for i in range(len(basis)):
    temp=0
    exec('X=X_{j}'.format(j=i))
    for j in range(len(basis)-1):
        pd=symbols(r'\partial_{u_{k}}'.replace('k',str(j+1)))
        temp+=X[j]*pd
    symb_reps.append(temp)
    
display('')
for a in symb_reps:
    display(simplify(a))

### Computing the Lie group of Symmetries 

In [27]:
mult_table

Matrix([
[   0, -e1/2, -e2/2, -e3/2,  -e4/2,               -e5/2,              -e6/2, -n],
[e1/2,     0,     0,     0,      0,                   0,                  n,  0],
[e2/2,     0,     0,     0,      0,                  -n,                  0,  0],
[e3/2,     0,     0,     0,      n,                   0,               a0*n,  0],
[e4/2,     0,     0,    -n,      0,               -a0*n,            -2*a1*n,  0],
[e5/2,     0,     n,     0,   a0*n,                   0, n*(a0**2 - a2 - b),  0],
[e6/2,    -n,     0, -a0*n, 2*a1*n, n*(-a0**2 + a2 + b),                  0,  0],
[   n,     0,     0,     0,      0,                   0,                  0,  0]])

In [35]:
a,b1,b2,b3,b4,b5,b6,n=symbols('a,b1,b2,b3,b4,b5,b6,n')
p_ad=ad_mat([a,b1,b2,b3,b4,b5,b6,n])

In [36]:
p_Ad=exp(p_ad)

In [53]:
simplify(p_Ad*ad_mats[0])

Matrix([
[0,                              0,                             0,                                     0,                                                   0,                                                                                                                0,                                                                                                                                    0,        0],
[0,                   -exp(-a/2)/2,                             0,                                     0,                                                   0,                                                                                                                0,                                                                                                                                    0,        0],
[0,                              0,                  -exp(-a/2)/2,                                     0,                                                  

### Tests and Checks

In [ ]:
Jacobi_check(mult_table,basis)

In [ ]:
# test of ad_mats

def commutator(a,b):
    return a*b-b*a

def Jac_expr(a,b,c):
    return simplify(commutator(A,commutator(B,C))
                    +commutator(B,commutator(C,A))+commutator(C,commutator(A,B)))

for i in range(len(ad_mats)):
    for j in range(i+1,len(ad_mats)):
        for k in range(j+1,len(ad_mats)):
            A,B,C=[ad_mats[i],ad_mats[j],ad_mats[k]]
            if simplify(Jac_expr(A,B,C))!=zeros(*shape(ad_mats[i])):print(i,j,k)

# general Jacobi test
y=zeros(*shape(x))
z=zeros(*shape(x))
for i in range(len(x)):
    exec('y_{j}=symbols("y_{j}")'.format(j=i))
    exec('z_{j}=symbols("z_{j}")'.format(j=i))
    exec('y[{j}]=y_{j}'.format(j=i))
    exec('z[{j}]=z_{j}'.format(j=i))

if simplify(Jac_expr(ad_mat(x),ad_mat(y),ad_mat(z)))!=zeros(*shape(ad_mats[0])):
    print('Failed general test')

In [ ]:
# X_i test
X_test=Matrix([list(Xi) for Xi in [X_0,X_1,X_2,X_3,X_4,X_5,X_6,X_7]]).transpose()-p_u
if X_test!= zeros(*shape(X_test)): print('Failed')

In [ ]:
# quick_coordinatize test

expr_list=[8*x_1-7*x_4,8*x_1-u_3*x_2,x_4+4*x_1-2*x_2]
for expr in expr_list:
    if (Matrix(quick_coordinatize(expr,x)).transpose()*x)[0,0]!=expr:
        print(expr,'-->',quick_coordinatize(expr,x))
        print(Matrix(quick_coordinatize(expr,x)).transpose()*x)

In [ ]:
# temp_ad test
for i in range(1,len(basis)):
    b=Matrix([0]*(len(basis)-1))
    b[i-1]=1
    exec('b{j}=copy.copy(b)'.format(j=i))

t1=(temp_ad(b1,u_1*b4)==b4)
t2=(temp_ad(b1,u_2*b4)==zeros(*shape(b1)))
t3=(temp_ad(u_5*b1,u_1*b4)==u_5*b4)
t4=(temp_ad(u_5*b1,u_1*b5)==u_5*b5-u_1*b1)
tests=[t1,t2,t3,t4]
for i in range(len(tests)):
    if not tests[i]: print('Failed t',i)